In [197]:
from pathlib import Path

import cv2
import numpy as np


ROOT = Path("C:\\Users\\rik_y\\Documents\\GitHub\\ricardoserrano.github.io\\Carcassone_Companion_App\\tile_library_z-man_2014")
SOURCE = ROOT / "crops"
DESTINATION = ROOT / "edge_classifications"
DESTINATION.mkdir(exist_ok=True)

In [198]:
IMGLIST = sorted(SOURCE.glob("IMG_*.jpg"))

In [199]:
def crop_tile(image: np.ndarray) -> tuple[np.ndarray, list[int]]:
    height, width = image.shape[:2]
    
    # 1. Convert BGR image to CIELAB color space
    lab = cv2.cvtColor(image, cv2.COLOR_BGR2Lab)
    _, a_channel, b_channel = cv2.split(lab)
    
    # 2. Calculate Chroma (distance from neutral gray at 128 in a* and b* channels)
    # This detects true color/artwork while ignoring shadows and pure white background.
    a_diff = a_channel.astype(np.float32) - 128.0
    b_diff = b_channel.astype(np.float32) - 128.0
    chroma = cv2.magnitude(a_diff, b_diff)
    
    # 3. Threshold chroma to isolate the tile artwork
    # Adjust '12' higher if tiny color noise triggers the mask, or lower for subtle pastel art
    _, mask = cv2.threshold(chroma.astype(np.uint8), 20, 255, cv2.THRESH_BINARY)

    # 4. Clean up minor noise and fill small interior gaps
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    
    # 5. Extract contours and find best tile candidate
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    center = np.array([width / 2.0, height / 2.0])
    choices = []
    
    for contour in contours:
        x, y, box_width, box_height = cv2.boundingRect(contour)
        area = box_width * box_height
        if area < 400:
            continue
        distance = np.linalg.norm(np.array([x + box_width / 2.0, y + box_height / 2.0]) - center)
        choices.append((distance - area / 2000.0, x, y, box_width, box_height))
        
    if not choices:
        raise ValueError("Could not isolate a tile")
        
    _, x, y, box_width, box_height = min(choices)
    
    # 6. Square bounding crop logic
    side = int(max(box_width, box_height))
    left = max(0, int(x + box_width / 2.0 - side / 2.0))
    top = max(0, int(y + box_height / 2.0 - side / 2.0))
    right = min(width, left + side)
    bottom = min(height, top + side)
    
    crop = image[top:bottom, left:right]
    
    
    # Keep output perfectly square if the crop hits image boundaries
    crop = cv2.copyMakeBorder(
        crop,
        0,
        side - crop.shape[0],
        0,
        side - crop.shape[1],
        cv2.BORDER_CONSTANT,
        value=(255, 255, 255),
    )
    
    return cv2.resize(crop, (512, 512), interpolation=cv2.INTER_AREA), [left, top, right, bottom]

In [200]:
def classify_segment(bgr_patch: np.ndarray) -> str:
    """Classifies an edge segment patch based on its HSV color profile."""
    hsv = cv2.cvtColor(bgr_patch, cv2.COLOR_BGR2HSV)
    
    # Define color ranges in OpenCV HSV (H: 0-180, S: 0-255, V: 0-255)
    green_mask = cv2.inRange(hsv, np.array([30, 180, 40]), np.array([50, 255, 255]))
    brown_mask = cv2.inRange(hsv, np.array([10, 140, 60]), np.array([30, 200, 140]))
    road_mask = cv2.inRange(hsv, np.array([0, 0, 120]), np.array([30, 60, 180]))

    total_pixels = bgr_patch.shape[0] * bgr_patch.shape[1]
    
    green_ratio = np.count_nonzero(green_mask) / total_pixels
    brown_ratio = np.count_nonzero(brown_mask) / total_pixels
    road_ratio = np.count_nonzero(road_mask) / total_pixels

    # Convert binary masks (1-channel) to BGR (3-channel) for concatenation
    green_bgr = cv2.cvtColor(green_mask, cv2.COLOR_GRAY2BGR)
    brown_bgr = cv2.cvtColor(brown_mask, cv2.COLOR_GRAY2BGR)
    road_bgr = cv2.cvtColor(road_mask, cv2.COLOR_GRAY2BGR)

    # Label each mask view directly on top
    font = cv2.FONT_HERSHEY_SIMPLEX
    cv2.putText(green_bgr, "Green", (5, 15), font, 0.4, (0, 255, 0), 1, cv2.LINE_AA)
    cv2.putText(brown_bgr, "Brown", (5, 15), font, 0.4, (0, 255, 255), 1, cv2.LINE_AA)
    cv2.putText(road_bgr, "Road", (5, 15), font, 0.4, (255, 255, 255), 1, cv2.LINE_AA)

    # Combine masks side-by-side
    masks_combined = cv2.hconcat([green_bgr, brown_bgr, road_bgr])

    # Add text panel for calculated ratios
    text_panel_height = 80
    text_panel = np.zeros((text_panel_height, bgr_patch.shape[1]), dtype=np.uint8)
    text_panel = cv2.cvtColor(text_panel, cv2.COLOR_GRAY2BGR)

    text_lines = [
        f"Green: {green_ratio:.2f}",
        f"Brown: {brown_ratio:.2f}",
        f"Road:  {road_ratio:.2f}"
    ]
    for i, line in enumerate(text_lines):
        cv2.putText(text_panel, line, (5, 20 + i * 20), font, 0.45, (255, 255, 255), 1, cv2.LINE_AA)

    # Stack original patch above the text panel
    top_section = cv2.vconcat([bgr_patch, text_panel])

    # Resize top section or mask bar to match widths before vertically stacking
    target_width = max(top_section.shape[1], masks_combined.shape[1])
    
    top_section_padded = cv2.copyMakeBorder(
        top_section, 0, 0, 0, target_width - top_section.shape[1], cv2.BORDER_CONSTANT, value=[0, 0, 0]
    )
    masks_padded = cv2.copyMakeBorder(
        masks_combined, 0, 0, 0, target_width - masks_combined.shape[1], cv2.BORDER_CONSTANT, value=[0, 0, 0]
    )

    # Final visual assembly: Top (Patch + Ratios) over Bottom (Masks)
    display_patch = cv2.vconcat([top_section_padded, masks_padded])

    cv2.imshow("Road Mask Analysis", display_patch)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

    # Classification Logic
    if road_ratio > 0.08:
        return "road"
    elif brown_ratio >= 0.12:
        return "city"
    elif green_ratio >= 0.3:
        return "field"
    else:
        return "unknown"

In [201]:
def crop_and_classify_tile(image: np.ndarray) -> tuple[np.ndarray, list[int], dict[str, list[str]]]:
    # 1. Crop the tile to a square and resize to 512x512
    tile_512, bounds = crop_tile(image)

    # 2. Extract 12 Edge Segments (3 per edge)
    # Depth defines how far inward into the tile edge to sample pixels (10% of size)
    patch_depth = int(512 * 0.05)
    # Distance defines how far from the top/bottom edge to start sampling (10% of size)
    patch_distance = int(512 * 0.05)
    # Width defines how far from the left/right edge to start sampling (10% of size)
    patch_width = int(512//3 * 0.02)

    step = 512 // 3  # ~170 pixels per segment
    
    segments = {"top": [], "right": [], "bottom": [], "left": []}

    for i in range(3):
        start_idx = i * step
        end_idx = (i + 1) * step if i < 2 else 512

        # Top Edge (3 patches along y: 0+patch_distance -> patch_depth+patch_distance)
        top_patch = tile_512[0+patch_distance:patch_depth+patch_distance, start_idx+patch_width:end_idx-patch_width]
        segments["top"].append(classify_segment(top_patch))

        # Right Edge (3 patches along x: 512-patch_depth-patch_distance -> 512-patch_distance)
        right_patch = tile_512[start_idx+patch_width:end_idx-patch_width, 512 - patch_depth - patch_distance:512 - patch_distance]
        segments["right"].append(classify_segment(right_patch))

        # Bottom Edge (3 patches along y: 512-patch_depth-patch_distance -> 512-patch_distance)
        bottom_patch = tile_512[512 - patch_depth - patch_distance:512 - patch_distance, start_idx+patch_width:end_idx-patch_width]
        segments["bottom"].append(classify_segment(bottom_patch))

        # Left Edge (3 patches along x: 0+patch_distance -> patch_depth+patch_distance)
        left_patch = tile_512[start_idx+patch_width:end_idx-patch_width, 0+patch_distance:patch_depth+patch_distance]
        segments["left"].append(classify_segment(left_patch))

    return tile_512, bounds, segments

In [202]:
image = cv2.imread(str(IMGLIST[1]))
# launch_tuning_gui(image)
tile_512, bounds, segments = crop_and_classify_tile(image)
print(segments)
cv2.imshow("Tile Crop", tile_512)
cv2.waitKey(0)
cv2.destroyAllWindows()

{'top': ['city', 'city', 'city'], 'right': ['city', 'road', 'city'], 'bottom': ['field', 'road', 'field'], 'left': ['field', 'field', 'field']}
